# Oblique shocks and Prandtl-meyer expansions with Gaslab

This notebook was translated from the original MATLAB live script. Examples that rely on plotting helpers or MATLAB-only convenience APIs not yet ported to Python are kept as notes or placeholders.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root_scalar

from gaslab import GasState
from gaslab.relations import arstar, oblique


© Tim Colonius, California Institute of Technology

Revised: July 11, 2023

Flow turning through an oblique shock or Prandtl-meyer exapnsion are invoked through the deflect process

In [2]:
help(GasState.deflect)


Help on function deflect in module gaslab.state:

deflect(self, ang: float, fam: int = 1, strong: bool = False)
    Turn a supersonic stream through an oblique shock or PM expansion.
    
    A positive deflection in the selected wave family corresponds to a
    compression and is solved as an oblique shock. A negative deflection
    corresponds to a Prandtl-Meyer expansion.
    
    Parameters
    ----------
    ang : float
        Signed deflection angle in radians, relative to the current flow
        direction.
    fam : int, default 1
        Wave family. Use ``+1`` for left-running waves and ``-1`` for
        right-running waves.
    strong : bool, default False
        If ``True``, use the strong oblique-shock branch when a shock
        solution exists.
    
    Returns
    -------
    GasState
        Deflected downstream state.
    
    Raises
    ------
    ValueError
        If the upstream flow is subsonic or if no valid shock/expansion
        solution exists for the req

Oblique shocks and Prandtl-Meyer expansions

An planar oblique shock is shown below. 

Analysis shows that the shock angle, \beta, is the solution of a nonlinear equation involving the deflection angle, \theta, and the upstream Mach number M.  The relation is

\tan\theta = 2 \cot\beta { ( M^2 \sin^2\beta- 1 ) 
 \over M^2(\gamma+\cos(2\beta))+2} 

Once the wave angle is determined, the solution for the downstream state follows from a normal shock analysis on the component of the flow normal to the oblique shock.  There are 0, 1, or 2 solutions for \beta for any given \theta and M.  When there are 2 solutions, these are called the weak (small \beta) and strong (large \beta) solutions.

In gaslab, a weak oblique shock is invoked with the 'deflect' process.  For example, to deflect a M=2 air stream by 10 degrees through a weak oblique shock, we use

In [3]:
GasState.init(2, 1.4).deflect(10*np.pi/180)


GasState(m=1.640522229001081, gamma=1.4, p=None, T=None, mw=None, p0=0.9846440225034329, t0=1.0, angle=0.17453292519943295, proc='oblique shock (39.31 deg)', dimmode=False, gcon=1.0, p0r=1.0, t0r=1.0, hist=GasState(m=2, gamma=1.4, p=None, T=None, mw=None, p0=1.0, t0=1.0, angle=0.0, proc='initstate', dimmode=False, gcon=1.0, p0r=1.0, t0r=1.0, hist=None, defaults=Defaults(small=1e-06, linewidth=2.0, fontsize=12, machrange=(0.01, 10.0), color=('b', 'r', 'g', 'k'))), defaults=Defaults(small=1e-06, linewidth=2.0, fontsize=12, machrange=(0.01, 10.0), color=('b', 'r', 'g', 'k')))

Notice in the output, the determined shock angle \beta is given in the process info.  

There are optional output arguments of deflect.  If we type

In [4]:
s = GasState.init(2, 1.4).deflect(10*np.pi/180)
bet, _, _ = oblique(2, 10*np.pi/180, 1.4)


the output 'bet' is set to the (weak) shock angle.  If we type

In [5]:
s = GasState.init(2, 1.4).deflect(10*np.pi/180)
bet, _, _ = oblique(2, 10*np.pi/180, 1.4)
# Theta-beta-M plot helper is not yet implemented in the Python port.


then we also get a visualization of the solution in the \theta -\beta -Mspace. 

Notice in the plot that for M=2, there is no solution beyond about 22 degrees.  The maximum deflection angle where there is a single solution can be found using 'thetamax' query:

In [6]:
180*GasState.init(2, 1.4).thetamax/np.pi


np.float64(22.97353176093794)

So if we try to deflect by more than 22.9735 degrees:

In [7]:
try:
    GasState.init(2, 1.4).deflect(30 * np.pi / 180)
except ValueError as exc:
    print(exc)


deflect: no oblique shock solution for the specified turn


The angle used in deflect is signed.  If we deflect the stream by -10 degrees, we will have a Prandtl-Meyer expansion rather than an oblique shock:

In [8]:
GasState.init(2, 1.4).deflect(-10 * np.pi / 180)  # PM expansion


GasState(m=2.38488715459307, gamma=1.4, p=None, T=None, mw=None, p0=1.0, t0=1.0, angle=-0.17453292519943295, proc='pm expansion', dimmode=False, gcon=1.0, p0r=1.0, t0r=1.0, hist=GasState(m=2, gamma=1.4, p=None, T=None, mw=None, p0=1.0, t0=1.0, angle=0.0, proc='initstate', dimmode=False, gcon=1.0, p0r=1.0, t0r=1.0, hist=None, defaults=Defaults(small=1e-06, linewidth=2.0, fontsize=12, machrange=(0.01, 10.0), color=('b', 'r', 'g', 'k'))), defaults=Defaults(small=1e-06, linewidth=2.0, fontsize=12, machrange=(0.01, 10.0), color=('b', 'r', 'g', 'k')))

There is an optional 3rd argument to deflect, which is the wave family, which we label as +1 or -1 depending onthat whether the wave is compressing the flow (through an oblique shock) through a positive or negative angle, respectively (or, for a Prandtl-meyer expansion, whether the flow is expanded through a negative or positive angle, respectively). This is depicted in the sketch below:

For many problems, the wave family can be omitted.

Thus each of the following produces the same state (apart from the flow angle):

In [9]:
GasState.init(2, 1.4).deflect(10*np.pi/180)
GasState.init(2, 1.4).deflect(10*np.pi/180, fam=1)
GasState.init(2, 1.4).deflect(-10*np.pi/180, fam=-1)


GasState(m=1.640522229001081, gamma=1.4, p=None, T=None, mw=None, p0=0.9846440225034329, t0=1.0, angle=-0.17453292519943295, proc='oblique shock (39.31 deg)', dimmode=False, gcon=1.0, p0r=1.0, t0r=1.0, hist=GasState(m=2, gamma=1.4, p=None, T=None, mw=None, p0=1.0, t0=1.0, angle=0.0, proc='initstate', dimmode=False, gcon=1.0, p0r=1.0, t0r=1.0, hist=None, defaults=Defaults(small=1e-06, linewidth=2.0, fontsize=12, machrange=(0.01, 10.0), color=('b', 'r', 'g', 'k'))), defaults=Defaults(small=1e-06, linewidth=2.0, fontsize=12, machrange=(0.01, 10.0), color=('b', 'r', 'g', 'k')))

We can obtain the strong shock solution (rarely needed in solving problems), by using the optional 4th argument to deflect:

In [10]:
# MATLAB multiple assignment not translated automatically:
# [s,bet,f] = deflect(gaslab(2,1.4), 10*pi/180, 1, 1); % logical true 4th argument gives strong shock


Note that the regions where solutions exist is bounded by the \theta =0 deflection (red lines).  From below, the week solutions are bounded by the Mach angle, \sin^{-1} {1 \over M}.  From above, the strong solutions are bounded by the normal shock solution.  

Problem 1 (Anderson ex. 4.1): A uniform stream of air at M_1 =3\ldotp 0,p_1= 1 atm, T_1=288K encounters a compression corner  that deflects the stream by 20^0.  Calculate the shock wave angle, and p_2, T_2, M_2, p_{02}, and T_{02} behind the shock wave.

In [11]:
os = GasState.init(3, 1.4, 1, 288, 28.96).deflect(np.pi * 20 / 180)
bet, _, _ = oblique(3, np.pi * 20 / 180, 1.4)
180 * bet / np.pi
os.pres
os.temp
os.mach
os.stagpres
os.stagtemp


806.4

These answer differ by a few percent from Anderson's solution.  

Note that the following two states are equivalent

In [12]:
os1 = GasState.init(3, 1.4, 1, 288, 28.96).deflect(np.pi*20/180, fam=1)
os2 = GasState.init(3, 1.4, 1, 288, 28.96).deflect(-np.pi*20/180, fam=-1)


Problem 2 (Under-expanded jet).  A supersonic air stream exits a planar CD nozzle at Mach 4 and expands to atmospheric pressure through a series of Prandtl-Meyer expansions and Oblique shocks (shock cells).  The situation near the nozzle exit depicted in the sketch.  The ratio {p_e\over p_a }= 10. Find M_d and \theta_d.

The key to this problem is understanding that in the region "d", the pressure must be matched with p_a across the slip line (vortex sheet).  Thus we want to find a P-M expansion from M_e through the angle \theta_d such that the downstream pressure decreases by 10.  Start by defining the exit state

In [13]:
exit = GasState.init(4, 1.4)


SyntaxError: invalid syntax (3549325532.py, line 1)

And make a function handle that returns the pressure ratio as a function of \theta_d:

In [ ]:
prat = lambda ang: exit.pres / (exit.deflect(ang)).pres


Solve for ang where prat = 10:

In [ ]:
ang = 180 * root_scalar(lambda ang: prat(ang) - 10, bracket=[-np.pi / 4, 0], method="brentq").root / np.pi


Now that we know the angle, find the flow in region "d" and display information about it:

In [ ]:
regd = exit.deflect(np.pi*ang/180)


Problem 3 (Lift and drag).  Consider a simple model of a supersonic airfoil in the form of a flat plate at an angle of attack \alpha.  For M=3, find and plot the lift and drag coefficients as a function of 0 < \alpha < 25^o.

The force normal to the plate is F_n = c (p_b - p_t )where c is the chord length.  Then normal force coefficient is then  C_{N} = { F_{N} \over {1 \over 2} \rho_\infty U_\infty^2 c } = { 2 F_{N} \over \gamma p_\infty M^2 c} = { 2  \over \gamma M^2 } \left( { p_b \over p_\infty} - {p_t \over p_\infty} \right)land ift and drag coefficients areC_D = C_N \sin\alpha, C_L = C_N \cos\alpha, respectively.  We just need to find the top and bottom pressure ratios.  Thus define the function

In [ ]:
cn = lambda alf: (
    GasState.init(3, 1.4).deflect(alf).pres - GasState.init(3, 1.4).deflect(-alf).pres
) / GasState.init(3, 1.4).pres


For M=3 and air, we can plot

In [ ]:
alfrange = np.pi * np.arange(0, 26, 1) / 180  # angles between 0 and 25 degrees
cn_vals = np.array([cn(a) for a in alfrange])
cd = np.sin(alfrange) * cn_vals
cl = np.cos(alfrange) * cn_vals
plt.figure()
plt.plot(180 * alfrange / np.pi, cd, label='C_D')
plt.plot(180 * alfrange / np.pi, cl, label='C_L')
plt.xlabel(r'$\alpha$')
plt.ylabel('Coefficient of force')
plt.legend()


Problem 4 (max deflection).  For air, make a plot of the deflection angle above which an oblique shock does not exist as a function of 0 < M < 4.

This problem can be solved directly by finding the thetamax property of states defined over the range of Mach numers:

In [ ]:
marange = np.linspace(1.1, 4, 20)
tmax = np.array([GasState.init(ma, 1.4).thetamax for ma in marange])
plt.fill_between(
    np.concatenate(([1], marange)),
    np.concatenate(([0], 180 * tmax / np.pi)),
    color='0.9',
    edgecolor='r',
)
plt.ylabel(r'$\theta$ (deg)')
plt.xlabel('M')
plt.text(3.0, 10.0, 'Oblique shock exists')
plt.text(1.5, 30.0, 'No solution')


Problem 5 (max deflection for regular reflection).  For air and the situation depicted in the sketch, if M = 3, find the maximum deflection angle \thetaabove which the assumed regular reflection cannot occur.

The flow processed by the first shock needs to be turned through the same angle with an oblique shock of the opposite family.  However, it's Mach number is reduced by the first shock, and therefore the maximum deflection angle for which the reflected shock exists is reduced.  We can find the condition as "what deflection angle produces an OS shock with a downstream flow that has a thetamax equal to that deflection angle?

In [ ]:
zerofun = lambda th: th - GasState.init(3, 1.4).deflect(th).thetamax
thmax = 180 * root_scalar(
    zerofun,
    x0=0.15,
    x1=0.2,
    method="secant",
).root / np.pi


Pressure-deflection diagrams

Pressure-deflection diagrams are useful for visualizing the solutions to many problems related to shock reflection and transmission.  For a given state (1), the diagram plots the pressure, relative to state 1, of all states reachable through oblique shocks and Prandtl-meyer expansions.  For example, for air at M=3, we have

In [ ]:
# Plot helper not yet implemented in the Python port:
# presdef(gaslab(3,1.4))


The figure below shows the anatomy of the pressure-deflection diagram.  

The diagram is especially useful for composite processes.  Consider problem 5 above.  Suppose the deflection angle \theta is {10}^0, which is less than the maximum for an oblique shock from M=4, and also less than the maximum deflection for the reflected wave.  Examine the pres-def diagram:

In [ ]:
# Plot helper not yet implemented in the Python port:
# presdef( deflect( gaslab(3,1.4), 10*pi/180))


 The reflected shock needs to turn the flow back to 0 degrees through a wave with family=-1.  This exists on the locus of possible states and so a regular reflection is possible.  However, let's increase the deflection now to {30}^0, which is less than the maximum for the first oblique shock, but greater than the maximum for the reflected wave:

In [ ]:
# Plot helper not yet implemented in the Python port:
# presdef( deflect( gaslab(3,1.4), 30*pi/180))


Notice now that there exists no oblique shock through which the flow in state 2 can be delected back by 30 degrees.  

Coming back to a {\theta \;=10}^0, consider the possibility of a 2nd reflection of the 1st-reflected wave from the ramp:

In [ ]:
# Plot helper not yet implemented in the Python port:
# presdef( deflect( deflect( gaslab(3,1.4), 10*pi/180), -10*pi/180, -1))


Apparently, yes, the flow can be deflected back from state 3 with another {+10}^0 deflection.  What about a 3rd reflection from the top wall?

In [ ]:
# Plot helper not yet implemented in the Python port:
# presdef( deflect( deflect( deflect( gaslab(3,1.4), 10*pi/180), -10*pi/180, -1), +10*pi/180, +1 ))


Yes, indeed, we can achieve a 3rd reflection!  Should we dare for a 4th?

In [ ]:
# Plot helper not yet implemented in the Python port:
# presdef( deflect( deflect( deflect( deflect( gaslab(3,1.4), 10*pi/180), -10*pi/180, -1), +10*pi/180, +1 ), -10*pi/180,-1))


Nope!  We see that in state 5 (this is the state downstream of the 3rd reflection, another deflection of {+10}^0 is not going to be possible through an oblique shock.  Thus a regular 4th reflection cannoth occur!  In case you're feeling dizzy, the following diagram might help:

Exercise: find the Mach number in regions 2 through 5 above.

In [ ]:
GasState.init(3, 1.4).deflect(10*np.pi/180).deflect(-10*np.pi/180, fam=-1).deflect(+10*np.pi/180, fam=+1).deflect(-10*np.pi/180, fam=-1)
